In [ ]:
from google.colab import drive
drive.mount('/content/drive')

ValueError: mount failed

In [ ]:
import pandas as pd

# Path sesuai dengan folder di Drive kamu
path_folder = '/content/drive/MyDrive/sentimen mbg dataset/'

# Membaca kedua file CSV
df1 = pd.read_csv(path_folder + 'MBG_sentiment_analysis.csv')
df2 = pd.read_csv(path_folder + 'komentar-mbg.csv')

# Cek 5 baris teratas dari masing-masing dataset
print("Dataset 1:")
display(df1.head())

print("\nDataset 2:")
display(df2.head())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

print("="*50)
print("1. INFORMASI DASAR DATASET 1 (MBG_sentiment_analysis)")
print("="*50)
print(df1.info())
print("\n--- Missing Values ---")
print(df1.isnull().sum())
print("\n--- Statistik Deskriptif ---")
display(df1.describe(include='all'))

print("\n" + "="*50)
print("2. INFORMASI DASAR DATASET 2 (komentar-mbg)")
print("="*50)
print(df2.info())
print("\n--- Missing Values ---")
print(df2.isnull().sum())
print("\n--- Statistik Deskriptif ---")
display(df2.describe(include='all'))

In [ ]:
import re

print("Jumlah awal Dataset 1:", len(df1))
print("Jumlah awal Dataset 2:", len(df2))

# ==========================================
# TAHAP 1: PREPROCESSING DATASET 1
# ==========================================
# Kita ambil kolom teks yang sudah bersih (misal: 'stopword removal' atau 'normalisasi')
# Di sini kita drop baris yang kosong (NaN)
df1_clean = df1.dropna(subset=['stopword removal']).copy()

# Kita buat format kolom yang seragam, misal ambil kolom 'stopword removal' sebagai teks akhir yang siapvektorisasi
df1_final = pd.DataFrame({
    'text_bersih': df1_clean['stopword removal'],
    'sumber': 'dataset_1'
})


# ==========================================
# TAHAP 2: PREPROCESSING DATASET 2 (Dari Mentah)
# ==========================================
def text_preprocessing(text):
    if pd.isna(text):
        return ""
    # 1. Case Folding (Ubah ke huruf kecil)
    text = str(text).lower()

    # 2. Cleaning (Hapus URL, mention, hashtag, angka, dan simbol/tanda baca)
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE) # Hapus URL
    text = re.sub(r'@\w+', '', text)    # Hapus mention
    text = re.sub(r'#\w+', '', text)    # Hapus hashtag
    text = re.sub(r'\d+', '', text)     # Hapus angka
    text = re.sub(r'[^\w\s]', '', text) # Hapus tanda baca/simbol

    # 3. Whitespace Removal (Hapus spasi berlebih)
    text = text.strip()
    return text

# Terapkan fungsi pembersihan ke kolom comment di Dataset 2
df2['text_bersih'] = df2['comment'].apply(text_preprocessing)

# Hapus baris yang teksnya jadi kosong setelah dibersihkan
df2_clean = df2[df2['text_bersih'] != ''].copy()

df2_final = pd.DataFrame({
    'text_bersih': df2_clean['text_bersih'],
    'sumber': 'dataset_2'
})


# ==========================================
# TAHAP 3: MENGGABUNGKAN KEDUA DATASET
# ==========================================
df_gabung = pd.concat([df1_final, df2_final], ignore_index=True)

print("\n" + "="*50)
print("HASIL AKHIR SETELAH DIGABUNG & DIBERSIHKAN")
print("="*50)
print(f"Jumlah baris Dataset 1 setelah dibersihkan: {len(df1_final)}")
print(f"Jumlah baris Dataset 2 setelah dibersihkan: {len(df2_final)}")
print(f"Total baris data gabungan siap pakai: {len(df_gabung)}")

# Cek 5 data teratas dari data gabungan
display(df_gabung.head())

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# 1. Pastikan semua teks diubah ke huruf kecil (Case Folding) & bersihkan dari karakter aneh
def full_cleaning(text):
    if pd.isna(text):
        return ""
    text = str(text).lower()
    # Hapus URL, mention, hashtag, angka, dan simbol
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    text = re.sub(r'@\w+|#\w+', '', text)
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'[^\w\s]', '', text)
    text = text.strip()
    return text

# Terapkan ke kolom text_bersih di df_gabung
df_gabung['text_bersih'] = df_gabung['text_bersih'].apply(full_cleaning)

# Hapus baris yang kosong setelah pembersihan menyeluruh
df_gabung = df_gabung[df_gabung['text_bersih'] != ''].reset_index(drop=True)

print(f"Total data setelah pembersihan menyeluruh: {len(df_gabung)} baris")

# 2. Feature Extraction: Mengubah teks bersih menjadi angka menggunakan TF-IDF
# TF-IDF ini penting agar teks bisa dibaca oleh algoritma Machine Learning
vectorizer = TfidfVectorizer(max_features=5000) # Batasi 5000 fitur kata teratas agar tidak terlalu berat
X_tfidf = vectorizer.fit_transform(df_gabung['text_bersih'])

print("\nBentuk matriks TF-IDF (Jumlah Baris, Jumlah Fitur Kata):", X_tfidf.shape)

In [ ]:
# Cek daftar nama seluruh kolom di df_gabung
print("Daftar Kolom di df_gabung:")
print(df_gabung.columns.tolist())

# Cek 5 baris pertama untuk melihat isi datanya secara utuh
display(df_gabung.head())

In [ ]:
import requests
import pandas as pd

print("Sedang mengunduh kamus leksikon positif & negatif dari GitHub...")

# 1. Unduh file raw positive.txt dan negative.txt langsung dari repo GitHub masdevid
url_pos = "https://raw.githubusercontent.com/masdevid/ID-OpinionWords/master/positive.txt"
url_neg = "https://raw.githubusercontent.com/masdevid/ID-OpinionWords/master/negative.txt"

res_pos = requests.get(url_pos)
res_neg = requests.get(url_neg)

# 2. Parsing teks per baris menjadi set kata (agar pencarian lebih cepat & unik)
positive_words = set(line.strip().lower() for line in res_pos.text.splitlines() if line.strip())
negative_words = set(line.strip().lower() for line in res_neg.text.splitlines() if line.strip())

print(f"Jumlah kata positif di kamus: {len(positive_words)}")
print(f"Jumlah kata negatif di kamus: {len(negative_words)}")

# 3. Fungsi pelabelan 3 kelas dengan tambahan Deteksi Negasi
def lexicon_labeling_advanced(text):
    if not isinstance(text, str):
        return 'Netral'

    words = text.split()
    pos_count = 0
    neg_count = 0

    # Daftar kata negasi bahasa Indonesia / slang
    negation_words = {'tidak', 'bukan', 'jangan', 'kurang', 'gak', 'nggak', 'ngga'}

    for i in range(len(words)):
        word = words[i]

        # Cek apakah kata sebelumnya adalah kata negasi
        is_negated = False
        if i > 0 and words[i-1] in negation_words:
            is_negated = True

        # Hitung dengan memperhitungkan apakah dibalik oleh kata negasi
        if word in positive_words:
            if is_negated:
                neg_count += 1  # Dibalik jadi negatif (contoh: "tidak bagus")
            else:
                pos_count += 1
        elif word in negative_words:
            if is_negated:
                pos_count += 1  # Dibalik jadi positif (contoh: "tidak buruk")
            else:
                neg_count += 1

    # Tentukan label berdasarkan dominasi skor akhir
    if pos_count > neg_count:
        return 'Positif'
    elif neg_count > pos_count:
        return 'Negatif'
    else:
        return 'Netral'

# 4. Terapkan fungsi pelabelan ke df_gabung milikmu
print("\nSedang memproses pelabelan otomatis dengan penanganan negasi pada df_gabung...")
df_gabung['sentiment_label'] = df_gabung['text_bersih'].apply(lexicon_labeling_advanced)

# 5. Mapping label teks ke bentuk angka (Label Encoding) untuk model BiLSTM
# 0: Negatif, 1: Netral, 2: Positif
label_mapping = {'Negatif': 0, 'Netral': 1, 'Positif': 2}
df_gabung['label'] = df_gabung['sentiment_label'].map(label_mapping)

# 6. Tampilkan hasil distribusi kelas sentimennya
print("\nDistribusi Label Sentimen Hasil Pelabelan (Revisi + Negation Handling):")
print(df_gabung['sentiment_label'].value_counts())

# Tampilkan sampel data teratas
display(df_gabung[['text_bersih', 'sentiment_label', 'label']].head(10))

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# 1. Menyiapkan fitur (X) dan target (y)
X = df_gabung['text_bersih'].astype(str).values
y = df_gabung['label'].values

# 2. Tokenisasi Teks (Mengubah kata menjadi urutan angka integer)
# max_words dibatasi misal 5000 kata unik yang paling sering muncul
max_words = 5000
tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")
tokenizer.fit_on_texts(X)

X_sequences = tokenizer.texts_to_sequences(X)

# 3. Padding (Menyamakan panjang seluruh kalimat agar seragam)
# max_len ditentukan misal 100 kata per kalimat
max_len = 100
X_padded = pad_sequences(X_sequences, maxlen=max_len, padding='post', truncating='post')

print("Bentuk data setelah tokenisasi & padding (X_padded):", X_padded.shape)

# 4. Train-Test-Split dengan proporsi 80:10:10 (Stratified agar distribusi kelas seimbang)
# Tahap 1: Pisah Train (80%) dan Sisa (20% untuk Val & Test)
X_train, X_temp, y_train, y_temp = train_test_split(
    X_padded, y, test_size=0.2, random_state=42, stratify=y
)

# Tahap 2: Bagi sisa 20% menjadi Validation (10%) dan Test (10%)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

# 5. Cek ukuran masing-masing subset data
print(f"Jumlah Data Latih (Train): {X_train.shape[0]} baris")
print(f"Jumlah Data Validasi (Val) : {X_val.shape[0]} baris")
print(f"Jumlah Data Uji (Test)     : {X_test.shape[0]} baris")

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout

# 1. Definisikan parameter model
vocab_size = min(5000, len(tokenizer.word_index) + 1)
embedding_dim = 100  # Dimensi vektor embedding FastText
max_len = 100        # Panjang maksimal sequence

# 2. Membangun Arsitektur Model FastText + BiLSTM
model = Sequential([
    # Layer Embedding (bisa diganti bobot FastText nanti, atau train from scratch sebagai baseline)
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_len),

    # Layer Bidirectional LSTM pertama dengan return_sequences=True untuk stacking
    Bidirectional(LSTM(64, return_sequences=True)),
    Dropout(0.3),

    # Layer Bidirectional LSTM kedua
    Bidirectional(LSTM(32)),
    Dropout(0.3),

    # Fully Connected Dense Layer untuk klasifikasi
    Dense(64, activation='relu'),
    Dropout(0.3),

    # Output Layer dengan 3 kelas (0: Negatif, 1: Netral, 2: Positif) menggunakan Softmax
    Dense(3, activation='softmax')
])

# 3. Kompilasi Model
model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

# Tampilkan ringkasan arsitektur model
model.summary()

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

# 1. Definisikan EarlyStopping agar training berhenti otomatis jika val_loss mulai stagnan/memburuk
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=3,          # Berhenti jika dalam 3 epoch berturut-turut tidak ada peningkatan
    restore_best_weights=True
)

print("Memulai proses training model FastText + BiLSTM...")

# 2. Proses Training (Fitting Model)
history = model.fit(
    X_train, y_train,
    epochs=10,             # Maksimal 10 epoch (biasanya cepat konvergen)
    batch_size=64,         # Ukuran batch per iterasi
    validation_data=(X_val, y_val),
    callbacks=[early_stopping],
    verbose=1
)

print("\nTraining Selesai! Model siap dievaluasi.")

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# 1. Melakukan prediksi pada data uji (X_test)
print("Menjalankan evaluasi pada Data Uji (Test Set)...")
y_pred_probs = model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)

# 2. Menampilkan Classification Report (Precision, Recall, F1-Score per kelas)
target_names = ['Negatif (0)', 'Netral (1)', 'Positif (2)']
print("\n--- CLASSIFICATION REPORT ---")
print(classification_report(y_test, y_pred, target_names=target_names))

# 3. Membuat Visualisasi Confusion Matrix
conf_matrix = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Negatif', 'Netral', 'Positif'],
            yticklabels=['Negatif', 'Netral', 'Positif'])
plt.title('Confusion Matrix - FastText + BiLSTM')
plt.xlabel('Prediksi Model')
plt.ylabel('Label Aktual (True Label)')
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# 1. Plot Grafik Accuracy (Akurasi Latih vs Validasi)
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy', linewidth=2)
plt.plot(history.history['val_accuracy'], label='Validation Accuracy', linewidth=2)
plt.title('Grafik Akurasi Model (FastText + BiLSTM)')
plt.xlabel('Epoch')
plt.ylabel('Akurasi')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)

# 2. Plot Grafik Loss (Loss Latih vs Validasi)
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss', color='orange', linewidth=2)
plt.plot(history.history['val_loss'], label='Validation Loss', color='red', linewidth=2)
plt.title('Grafik Loss Model (FastText + BiLSTM)')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

In [ ]:
import pickle

# 1. Menyimpan model utama ke format .keras (standar baru TensorFlow/Keras)
model.save("model_sentimen_mbg.keras")
print("Model berhasil disimpan sebagai 'model_sentimen_mbg.keras'")

# 2. Menyimpan tokenizer menggunakan pickle agar pemetaan kata dan angkanya tetap sinkron
with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)
print("Tokenizer berhasil disimpan sebagai 'tokenizer.pkl'")

In [ ]:
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences
import pickle
import numpy as np

# 1. Load kembali model dan tokenizer yang sudah disimpan sebelumnya
loaded_model = load_model("model_sentimen_mbg.keras")
with open("tokenizer.pkl", "rb") as f:
    loaded_tokenizer = pickle.load(f)

# 2. Masukkan kalimat uji terserah kamu di sini!
kalimat_baru = [
    "aku suka mbg tapi ga suka prabowo",
    "prabowo kdrt sama teddy",
    "prabowo kokop teddy"
]

# 3. Pra-proses kalimat (tokenisasi & padding sesuai parameter saat training)
max_len = 100
sequences = loaded_tokenizer.texts_to_sequences(kalimat_baru)
padded = pad_sequences(sequences, maxlen=max_len, padding='post', truncating='post')

# 4. Lakukan Prediksi
predictions = loaded_model.predict(padded)
predicted_labels = np.argmax(predictions, axis=1)

# Mapping angka kembali ke label teks
label_map = {0: 'Negatif', 1: 'Netral', 2: 'Positif'}

# 5. Tampilkan Hasilnya
print("\n--- HASIL PREDIKSI KALIMAT BARU ---")
for teks, label_idx in zip(kalimat_baru, predicted_labels):
    print(f"Teks   : {teks}")
    print(f"Prediksi Sentimen : {label_map[label_idx]}")
    print("-" * 40)